# Example queries: `applied_only` (resstock_oedi)

Auto-generated from `tests/query_snapshots/applied_only.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [ ]:
from pathlib import Path
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [ ]:
# This notebook lives in `tests/example_notebooks/`; the snapshot test
# cache is its sibling `tests/query_snapshots/resstock_oedi_cache/`. Resolve
# the path relative to the notebook directory (`_dh[0]` is set by
# IPython at kernel startup; falls back to CWD outside Jupyter).
_NB_DIR = Path(_dh[0] if "_dh" in globals() else ".").resolve()
_CACHE = (_NB_DIR / "../query_snapshots/resstock_oedi_cache").resolve()
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "resstock_2024_amy2018_release_2",
    buildstock_type="resstock",
    db_schema="resstock_oedi_vu",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


## `applied_only_upgrade1_annual`

Annual upgrade 1 electricity restricted to applied buildings only, CO.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    upgrade_id='1',
    group_by=['geometry_building_type_recs'],
    restrict=[('state', ['CO'])],
    applied_only=True,
)
result.head() if hasattr(result, 'head') else result


# shape: (5, 4)
  geometry_building_type_recs  sample_count  units_count  electricity.total.energy_consumption
                  Mobile Home           370 9.335161e+04                          1.182440e+09
Multi-Family with 2 - 4 Units           467 1.178249e+05                          9.854143e+08
   Multi-Family with 5+ Units          1872 4.723087e+05                          3.070324e+09
       Single-Family Attached           662 1.670237e+05                          1.803582e+09
       Single-Family Detached          5700 1.438119e+06                          2.651883e+10


## `all_buildings_upgrade1_annual`

Annual upgrade 1 including unapplied buildings, CO. Two variants — upgrade_id as string '1' vs int 1 — must compile to the same SQL since _validate_upgrade normalizes both to str.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    upgrade_id='1',
    group_by=['geometry_building_type_recs'],
    restrict=[('state', ['CO'])],
    applied_only=False,
)
result.head() if hasattr(result, 'head') else result


# shape: (5, 4)
  geometry_building_type_recs  sample_count  units_count  electricity.total.energy_consumption
                  Mobile Home           391 9.864994e+04                          1.218917e+09
Multi-Family with 2 - 4 Units           469 1.183295e+05                          9.869626e+08
   Multi-Family with 5+ Units          2001 5.048556e+05                          3.258222e+09
       Single-Family Attached           664 1.675283e+05                          1.809644e+09
       Single-Family Detached          5900 1.488580e+06                          2.693151e+10


## `include_baseline_upgrade1_annual`

Annual upgrade 1 with include_baseline=true and include_savings=false. Mirrors the comstock notebook pattern that exposes baseline + upgrade columns side-by-side without the savings subtraction.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    upgrade_id='1',
    group_by=['geometry_building_type_recs'],
    restrict=[('state', ['CO'])],
    include_baseline=True,
    include_upgrade=True,
    include_savings=False,
    applied_only=False,
)
result.head() if hasattr(result, 'head') else result


# shape: (5, 5)
  geometry_building_type_recs  sample_count  units_count  electricity.total.energy_consumption__baseline  electricity.total.energy_consumption__upgrade
                  Mobile Home           391 9.864994e+04                                    7.898215e+08                                   1.218917e+09
Multi-Family with 2 - 4 Units           469 1.183295e+05                                    8.547447e+08                                   9.869626e+08
   Multi-Family with 5+ Units          2001 5.048556e+05                                    3.249347e+09                                   3.258222e+09
       Single-Family Attached           664 1.675283e+05                                    1.345615e+09                                   1.809644e+09
       Single-Family Detached          5900 1.488580e+06                                    1.635437e+10                                   2.693151e+10


## `baseline_with_applied_in_upgrade1`

Annual baseline (upgrade_id=0) restricted via applied_in=[1] — buildings where upgrade 1 applied successfully. Validator was relaxed in 2026-04-26 commit (e85e741) to allow applied_in on baseline queries; before that, the validator rejected this combination as 'applied_in cannot be set when applied_only is False'. Pins the SQL shape that test_savings_equals_independent_baseline_minus_upgrade depends on.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    upgrade_id='0',
    applied_in=[1],
    group_by=['geometry_building_type_recs'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


# shape: (5, 4)
  geometry_building_type_recs  sample_count  units_count  electricity.total.energy_consumption
                  Mobile Home           370 9.335161e+04                          7.533442e+08
Multi-Family with 2 - 4 Units           467 1.178249e+05                          8.531964e+08
   Multi-Family with 5+ Units          1872 4.723087e+05                          3.061449e+09
       Single-Family Attached           662 1.670237e+05                          1.339553e+09
       Single-Family Detached          5700 1.438119e+06                          1.594169e+10


## `include_baseline_upgrade1_applied_only_annual`

Same as include_baseline_upgrade1_annual but with applied_only=true. Hits the upgrade_only=False branch (because include_baseline forces the baseline join).


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    upgrade_id='1',
    group_by=['geometry_building_type_recs'],
    restrict=[('state', ['CO'])],
    include_baseline=True,
    include_upgrade=True,
    include_savings=False,
    applied_only=True,
)
result.head() if hasattr(result, 'head') else result


# shape: (5, 5)
  geometry_building_type_recs  sample_count  units_count  electricity.total.energy_consumption__baseline  electricity.total.energy_consumption__upgrade
                  Mobile Home           370 9.335161e+04                                    7.533442e+08                                   1.182440e+09
Multi-Family with 2 - 4 Units           467 1.178249e+05                                    8.531964e+08                                   9.854143e+08
   Multi-Family with 5+ Units          1872 4.723087e+05                                    3.061449e+09                                   3.070324e+09
       Single-Family Attached           662 1.670237e+05                                    1.339553e+09                                   1.803582e+09
       Single-Family Detached          5700 1.438119e+06                                    1.594169e+10                                   2.651883e+10
